# document_ingestion

In [5]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

file_path = r"C:\Users\DhananjaiSingh\Desktop\PythonTutorials\Gen-AI\input\lambda-dg.pdf"
def load_pdf(file_path: os.path) -> str:
    """
    """
    
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    all_texts = ''
    for doc in docs:
        all_texts = all_texts + doc.page_content
    return all_texts


def create_langchain_docs(text:str) -> List[Document]:
    """
    """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    texts = text_splitter.split_text(text)
    document_list = []
    for txt in texts:
        metadata = {'file_name':'AWS_Lambda documentation'}
        document_list.append(Document(page_content=txt, metadata = metadata))
    return document_list


In [6]:
pdf_content = load_pdf(file_path)
pdf_docs = create_langchain_docs(pdf_content)

In [ ]:
pdf_docs[0]

# Vector Stores

In [1]:
from services.llm_service import embeddings_model as embedding
from langchain_chroma import Chroma
from typing import List

def create_vector_store(collection_name:str) -> Chroma:
    vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embedding,
    persist_directory="./chroma_langchain_db",
    )
    return vector_store
    

import chromadb
from langchain_chroma import Chroma
from services.llm_service import embeddings_model as embedding

def add_documents_vectorDB(collection_name: str, documents:List ):
    client = chromadb.PersistentClient(path="./chroma_langchain_db")
    vector_store_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding,)

    vector_store_from_client.add_documents(documents)




In [2]:
vector_store = create_vector_store(collection_name='AWS_Lambda')
results = vector_store.similarity_search(
    "What is AWS Lambda?",
    k=2,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Developer Guide
AWS Lambda
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.AWS Lambda
Developer Guide
AWS Lambda: Developer Guide
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.
Amazon's trademarks and trade dress may not be used in connection with any product or service 
that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any 
manner that disparages or discredits Amazon. All other trademarks not owned by Amazon are 
the property of their respective owners, who may or may not be aﬃliated with, connected to, or 
sponsored by Amazon.AWS Lambda
Developer Guide
Table of Contents
What is AWS Lambda? .................................................................................................................... 1
When to use Lambda .................................................................................................................................. 1 [{'file_nam

In [ ]:
results = vector_store.similarity_search_with_score(
    "What is AWS Lambda?", k=2,
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.253119] Developer Guide
AWS Lambda
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.AWS Lambda
Developer Guide
AWS Lambda: Developer Guide
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.
Amazon's trademarks and trade dress may not be used in connection with any product or service 
that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any 
manner that disparages or discredits Amazon. All other trademarks not owned by Amazon are 
the property of their respective owners, who may or may not be aﬃliated with, connected to, or 
sponsored by Amazon.AWS Lambda
Developer Guide
Table of Contents
What is AWS Lambda? .................................................................................................................... 1
When to use Lambda .................................................................................................................................

In [ ]:
#   (Maximal Marginal Relevance)

retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 2, "fetch_k": 5}
)

retriever.invoke("What is AWS Lambda?")


[Document(id='5977394c-481d-4be9-9c4a-eabc7c50bb83', metadata={'file_name': 'AWS_Lambda documentation'}, page_content="Developer Guide\nAWS Lambda\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.AWS Lambda\nDeveloper Guide\nAWS Lambda: Developer Guide\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \nthat is not Amazon's, in any manner that is likely to cause confusion among customers, or in any \nmanner that disparages or discredits Amazon. All other trademarks not owned by Amazon are \nthe property of their respective owners, who may or may not be aﬃliated with, connected to, or \nsponsored by Amazon.AWS Lambda\nDeveloper Guide\nTable of Contents\nWhat is AWS Lambda? .................................................................................................................... 1\nWhen to use Lambda .............

In [8]:
result = ''
for res in results:
    result = result + '\n\n' + res.page_content


In [7]:
result

"Developer Guide\nAWS Lambda\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.AWS Lambda\nDeveloper Guide\nAWS Lambda: Developer Guide\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \nthat is not Amazon's, in any manner that is likely to cause confusion among customers, or in any \nmanner that disparages or discredits Amazon. All other trademarks not owned by Amazon are \nthe property of their respective owners, who may or may not be aﬃliated with, connected to, or \nsponsored by Amazon.AWS Lambda\nDeveloper Guide\nTable of Contents\nWhat is AWS Lambda? .................................................................................................................... 1\nWhen to use Lambda .................................................................................................................................. 

In [ ]:
from services.llm_service import llm


In [5]:
llm.invoke("What is capital of India?")

AIMessage(content='The capital of India is **New Delhi**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--c14afd7b-854e-4b4e-b665-5e2305060979-0', usage_metadata={'input_tokens': 6, 'output_tokens': 10, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

# Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template  = ChatPromptTemplate([
    ("system", "You Based on the provided context, give the answer to foloowing question {context}"),
    ("user", "{user_query}")
])

prompt_template.invoke({"user_query": 'What is AWS Lambda?' , "context":result })

ChatPromptValue(messages=[SystemMessage(content="You are a helpful assistant. Based on the provided context, give the answer to foloowing question \n\nDeveloper Guide\nAWS Lambda\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.AWS Lambda\nDeveloper Guide\nAWS Lambda: Developer Guide\nCopyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \nthat is not Amazon's, in any manner that is likely to cause confusion among customers, or in any \nmanner that disparages or discredits Amazon. All other trademarks not owned by Amazon are \nthe property of their respective owners, who may or may not be aﬃliated with, connected to, or \nsponsored by Amazon.AWS Lambda\nDeveloper Guide\nTable of Contents\nWhat is AWS Lambda? .................................................................................................................... 1\n

In [13]:
llm_chain = prompt_template | llm

In [15]:
llm_chain.invoke({"user_query": "What is AWS Lambda?", "context":result })

AIMessage(content='According to the provided document, AWS Lambda is introduced in the "What is AWS Lambda?" section, which starts on page 1.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--8e9e1827-59a0-4491-9007-776ed6fcdacf-0', usage_metadata={'input_tokens': 324, 'output_tokens': 28, 'total_tokens': 352, 'input_token_details': {'cache_read': 0}})

In [17]:
llm.invoke("India is a country situated in Asia. FRom this statement extract the names entity. Give the the response in the following structure: Entity 1: <Entity>, Entity 2: <Entity>, and so on  ")

AIMessage(content='Entity 1: India, Entity 2: Asia', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--d2a69da8-c1fc-4343-a8c7-42bb76105740-0', usage_metadata={'input_tokens': 44, 'output_tokens': 12, 'total_tokens': 56, 'input_token_details': {'cache_read': 0}})

In [ ]:
'''
1 Clear instruction 
2 Provice context 
'''


In [20]:
essay1 = '''The cow is a gentle animal that gives us milk.
It has four legs, two horns, and eats grass.'''

essay2 = '''The cow is a useful domestic animal that provides us with milk, which is rich in nutrients.
It plays an important role in agriculture and is respected in many cultures for its gentle nature.'''

print(llm.invoke(f"""Evaluate the folloing essay on the scale of 5 where 5 is best and 1 is worst. 
                 Assume this essay is written by class 2 student. The essay is: {essay1}. 
                 Give the response in following format:
                   ```json
                    score: <score>,
                    advantages: <positive points in the essay>,
                    disadvantages: <any shortcomings in the essay>
                ```  """))

content='```json\n{\n  "score": 4,\n  "advantages": [\n    "Simple and clear language appropriate for a class 2 student.",\n    "Identifies key characteristics of a cow (milk, legs, horns, diet).",\n    "States a positive attribute (gentle animal)."\n  ],\n  "disadvantages": [\n    "Could be more descriptive (e.g., color, size).",\n    "Lacks detail and elaboration. Each point is stated simply without further explanation.",\n    "Sentence structure is very basic and repetitive."\n  ]\n}\n```' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []} id='run--0bf7975d-1265-451b-8e45-35cb274ec517-0' usage_metadata={'input_tokens': 113, 'output_tokens': 127, 'total_tokens': 240, 'input_token_details': {'cache_read': 0}}


# Agents

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain.tools import Tool
from langchain.chains.llm_math.base import LLMMathChain
from dotenv import load_dotenv
import time

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

6

In [ ]:
from langchain_core.tools import tool


@tool
def websearch(query: str) -> str:
    '''
    Used to perform the web search using DuckDuckGo
    '''
    # Implememnt you r logic here 
    return len(query)*1000


In [ ]:



#  Math Tool: LLMMathChain for mathematical calculations
# This chain uses the LLM to parse the math problem and then a numerical expression evaluator.
llm_math_chain = LLMMathChain.from_llm(llm=llm, verbose=True)
math_tool = Tool(
    name="Calculator",
    func=llm_math_chain,
    description="Useful for when you need to answer questions about math. Input should be a mathematical expression string (e.g., '2 + 2', 'sqrt(16)')."
)

# Combine our tools into a list
tools = [math_tool , websearch ]

print("Tools defined successfully!")

In [ ]:
prompt = hub.pull("hwchase17/react")

# Create the ReAct agent
agent = create_react_agent(llm, tools, prompt)

print("Agent created!")

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # This is crucial to see the Thought-Action-Observation loop
    handle_parsing_errors=True # Helps the agent recover from malformed LLM outputs
)

print("AgentExecutor initialized!")
# Python Code Example 5: Run the Agent!
print("\n--- Running Agent Query 1 ---")
query1 = "What is population of France? What is that number multiplied by 0.05?"
result1 = agent_executor.invoke({"input": query1})
print(f"\nFinal Answer 1: {result1['output']}")

